# Attention, Positional Encoding, Encoder Block
- https://newsletter.theaiedge.io/p/the-transformer-architecture-v2

### 1. Attention Mechanism


**Objective**: Enable models to focus on relevant parts of input sequences, essential for handling longer dependencies.

**Explanation**: 
- Attention allows a model to “attend” to specific input parts when generating each output. It’s especially useful for tasks requiring selective referencing of input tokens (e.g., translation, summarization).

    

In [16]:
import torch
import torch.nn as nn

# Simple attention mechanism
def attention(query, key, value):
    scores = torch.matmul(query, key.transpose(-2, -1)) / torch.sqrt(torch.tensor(key.size(-1), dtype=torch.float32))
    attn_weights = torch.nn.functional.softmax(scores, dim=-1)
    output = torch.matmul(attn_weights, value)
    return output, attn_weights

# Example inputs
query = torch.randn(5, 3, 8)  # Batch size = 5, Sequence length = 3, Embedding size = 8
key = torch.randn(5, 3, 8)
value = torch.randn(5, 3, 8)

# Apply attention
output, weights = attention(query, key, value)
print("Attention Output:", output)
print("Attention Weights:", weights)
    

Attention Output: tensor([[[-0.4943,  0.2761, -0.2303, -0.8255,  0.3338, -1.2909,  0.2104,
           0.5830],
         [-0.2107,  0.5070, -0.4783, -0.8237,  0.5107, -1.0102,  0.2253,
           0.4435],
         [-0.9002, -0.1110,  0.2886, -0.5817,  0.0136, -1.7752,  0.1259,
           1.0911]],

        [[-0.9302,  1.3205,  0.0547,  0.3881, -0.6762,  0.0052,  0.4088,
           0.7115],
         [ 0.2799,  0.0214, -0.1624,  0.0784,  2.1545,  0.7974,  0.4849,
          -0.1573],
         [-0.0488,  0.6487, -0.5219, -0.0573,  0.2955,  0.4514,  1.2375,
           0.6065]],

        [[ 0.3241,  0.7496,  0.2037, -1.1759,  0.2625, -0.6728,  0.4299,
          -0.9504],
         [ 0.1020,  0.6815,  0.1122, -1.1616,  0.2738, -0.7349,  0.5698,
          -0.8907],
         [ 0.3810,  1.1424,  0.4728, -1.1883,  0.0972, -0.3462,  0.3684,
          -0.8580]],

        [[ 1.3299, -0.1410,  0.3767, -0.6937,  0.2161, -0.3268, -0.3123,
          -0.8652],
         [ 0.9943, -0.3060,  0.5642, -0.9367, 

### 2. Self-Attention Mechanism


**Objective**: Understand how self-attention allows a model to relate each word in a sentence to every other word, helping capture context effectively.

**Explanation**:
   - **Queries, Keys, and Values**: In self-attention, each word is represented by three vectors: a query, a key, and a value.
   - **Dot Product and Scaling**: The query and key vectors are multiplied to determine attention scores, representing the similarity between words. These scores are scaled and then passed through a softmax function to obtain attention weights.
   - **Weighted Sum**: Each word’s final representation is the sum of all value vectors, weighted by attention scores, allowing the model to focus on relevant words in context.
    

In [17]:

import torch
import torch.nn.functional as F

# Self-attention function
def self_attention(query, key, value):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))
    attn_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attn_weights, value)
    return output, attn_weights

# Example inputs
query = torch.randn(2, 4, 8)  # Batch size = 2, Sequence length = 4, Embedding size = 8
key = torch.randn(2, 4, 8)
value = torch.randn(2, 4, 8)

# Apply self-attention
output, attn_weights = self_attention(query, key, value)
print("Self-Attention Output:", output)
print("Attention Weights:", attn_weights)
    

Self-Attention Output: tensor([[[-0.0077,  0.5413,  0.4530,  0.0170,  0.8612, -0.7753, -0.9428,
          -0.5993],
         [ 0.2523, -0.1573,  0.9531,  0.1938, -0.2793,  0.0189,  0.0459,
          -0.6005],
         [ 0.3904, -0.3326,  1.0638,  0.1339, -0.7248,  0.2699,  0.2203,
          -0.5354],
         [ 0.1172, -0.0399,  0.3471, -0.6320,  0.4307, -0.8574, -1.6011,
           0.3489]],

        [[-0.0233,  0.6465, -0.4277,  0.4091,  0.3735,  0.3476,  0.2495,
           0.7046],
         [-0.0659,  0.0671, -0.5161, -1.3166,  0.8776, -0.2364,  0.1346,
          -0.3521],
         [-0.4777,  0.3455, -0.2992,  0.2823,  0.1967,  0.1292,  0.5089,
           0.4893],
         [ 0.0276,  0.3360,  0.4090, -0.8555, -0.0176, -0.1098,  0.3742,
          -0.6784]]])
Attention Weights: tensor([[[0.1527, 0.2815, 0.0523, 0.5134],
         [0.0773, 0.0401, 0.6845, 0.1981],
         [0.0447, 0.0211, 0.8719, 0.0623],
         [0.3226, 0.5061, 0.1384, 0.0329]],

        [[0.5149, 0.1104, 0.2889, 0.

### 3. Multi-Head Attention


**Objective**: Enhance the model’s ability to capture diverse word relationships by using multiple attention heads.

**Explanation**:
   - **Multiple Heads**: Multi-head attention applies self-attention multiple times in parallel, each with different learned projections. This allows the model to capture various types of dependencies, such as grammatical or semantic.
   - **Concatenation and Linear Transformation**: Each head’s output is concatenated and passed through a linear layer, blending the insights from all heads into a single representation.
    

In [18]:

import torch.nn as nn

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_size, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert embed_size % num_heads == 0
        self.head_dim = embed_size // num_heads
        self.num_heads = num_heads

        # Linear layers for queries, keys, values
        self.query = nn.Linear(embed_size, embed_size)
        self.key = nn.Linear(embed_size, embed_size)
        self.value = nn.Linear(embed_size, embed_size)
        
        # Final linear layer after concatenation
        self.fc_out = nn.Linear(embed_size, embed_size)

    def forward(self, x):
        batch_size, seq_len, embed_size = x.size()

        # Transform inputs into multiple heads
        query = self.query(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        key = self.key(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        value = self.value(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Apply self-attention
        scores = torch.matmul(query, key.transpose(-2, -1)) / torch.sqrt(torch.tensor(self.head_dim, dtype=torch.float32))
        attn_weights = F.softmax(scores, dim=-1)
        attention = torch.matmul(attn_weights, value)
        
        # Concatenate heads and pass through the final linear layer
        attention = attention.transpose(1, 2).contiguous().view(batch_size, seq_len, embed_size)
        return self.fc_out(attention)

# Example usage
embed_size = 16
num_heads = 4
x = torch.randn(2, 5, embed_size)  # Batch size = 2, Sequence length = 5

multi_head_attention = MultiHeadAttention(embed_size, num_heads)
output = multi_head_attention(x)
print("Multi-Head Attention Output:", output)
    

Multi-Head Attention Output: tensor([[[-3.0796e-01,  9.9516e-03,  3.3698e-01, -2.6343e-01, -7.6173e-02,
           8.6556e-02, -1.6535e-01, -6.4851e-02, -1.4225e-01, -3.5264e-01,
           1.2869e-01, -6.2537e-03,  6.0203e-01,  2.5822e-02,  3.9249e-01,
           5.0380e-02],
         [-3.0499e-01, -1.7136e-02,  3.2240e-01, -2.3885e-01, -4.0089e-02,
           9.7080e-02, -1.5485e-01, -8.0591e-02, -1.3564e-01, -3.8821e-01,
           1.4603e-01,  3.0483e-02,  5.5505e-01,  4.4241e-02,  3.4218e-01,
           2.0446e-03],
         [-3.0866e-01,  2.9959e-02,  3.3191e-01, -1.9964e-01, -4.8877e-02,
           9.9783e-02, -1.9272e-01, -4.6513e-02, -1.2934e-01, -3.9473e-01,
           1.5140e-01,  4.7113e-02,  5.3599e-01,  1.7858e-02,  3.6926e-01,
          -1.2505e-02],
         [-3.1347e-01,  2.9241e-02,  3.0913e-01, -1.8524e-01, -1.3395e-02,
           5.2456e-02, -2.5054e-01, -6.7783e-02, -7.8303e-02, -3.7788e-01,
           1.3992e-01,  1.1102e-01,  5.9619e-01,  4.1949e-02,  4.4681e-01,

### 4. Positional Encoding


**Objective**: Introduce positional information into word embeddings, as Transformers process words in parallel without inherent sequence order.

**Explanation**:
   - Since Transformers do not process words sequentially, positional encodings are added to the embeddings to give the model information about the order of words.
   - **Sine and Cosine Functions**: Positional encodings are computed using sine and cosine functions with varying frequencies, creating unique patterns for each position in the sequence.
    

In [19]:

import math

class PositionalEncoding(nn.Module):
    def __init__(self, embed_size, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, embed_size)
        for pos in range(max_len):
            for i in range(0, embed_size, 2):
                pe[pos, i] = math.sin(pos / (10000 ** ((2 * i) / embed_size)))
                pe[pos, i + 1] = math.cos(pos / (10000 ** ((2 * i) / embed_size)))
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return x

# Example usage
embed_size = 16
seq_len = 10
pos_encoding = PositionalEncoding(embed_size)
x = torch.randn(1, seq_len, embed_size)
output = pos_encoding(x)
print("Positional Encoding Output:", output)
    

Positional Encoding Output: tensor([[[-0.6863, -0.1787,  0.2665,  1.9848,  0.4873,  0.7678,  0.9001,
           0.7125, -1.1633, -0.0557, -0.8258,  1.6015,  1.9631,  2.2323,
          -0.5906,  0.6126],
         [-0.7676, -1.3410, -0.8735,  1.5396,  0.9730, -1.5752, -0.4326,
           2.0249, -1.3086,  2.8208,  1.0821,  1.7148,  0.4493,  2.3523,
           0.5511,  0.5643],
         [-0.2211,  1.7056, -0.1696,  0.7574,  0.7759,  1.1884, -1.5574,
           0.1605, -0.4072,  2.2335,  0.7148,  0.3693,  1.9091,  1.2612,
          -1.4521,  0.6459],
         [-1.9246,  1.8145,  2.6719,  0.3800, -0.3320,  2.3965,  0.7526,
           2.1607,  0.0202,  0.0162, -0.8506,  2.9702, -0.0040,  1.9019,
           1.8104,  0.7881],
         [-0.9294, -1.3841, -0.0861,  1.2240, -0.2229,  0.8896, -1.0953,
           2.5122, -1.4364,  0.3048, -0.1532,  1.8307,  0.7987, -1.3276,
           0.2446,  1.6985],
         [-1.5727,  0.0948,  0.9341,  2.1655, -0.5877,  1.8663, -1.5414,
           2.4860, -1.79

### 4. Transformer Encoder Block


**Objective**: Understand how the Transformer encoder combines self-attention and feedforward layers, the fundamental units in a Transformer.

**Explanation**:
   - **Self-Attention Layer**: The encoder starts with a multi-head attention layer, allowing the model to attend to relevant parts of the input.
   - **Add & Norm**: After self-attention, a residual connection is added, followed by layer normalization to stabilize learning.
   - **Feedforward Layer**: The attention output is passed through a fully connected layer for further processing.
   - **Final Add & Norm**: A second residual connection and layer normalization complete the block, making the Transformer’s encoder robust to varying input sequences.
    

In [20]:

class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_size, num_heads, forward_expansion):
        super(TransformerEncoderBlock, self).__init__()
        self.attention = MultiHeadAttention(embed_size, num_heads)
        self.norm1 = nn.LayerNorm(embed_size)
        self.norm2 = nn.LayerNorm(embed_size)

        self.feed_forward = nn.Sequential(
            nn.Linear(embed_size, forward_expansion * embed_size),
            nn.ReLU(),
            nn.Linear(forward_expansion * embed_size, embed_size)
        )

    def forward(self, x):
        attention = self.attention(x)
        x = self.norm1(attention + x)  # Add & Norm
        forward = self.feed_forward(x)
        x = self.norm2(forward + x)  # Add & Norm
        return x

# Example usage
embed_size = 16
num_heads = 4
forward_expansion = 4
encoder_block = TransformerEncoderBlock(embed_size, num_heads, forward_expansion)
x = torch.randn(2, 5, embed_size)
output = encoder_block(x)
print("Transformer Encoder Block Output:", output)
    

Transformer Encoder Block Output: tensor([[[-1.8988, -0.4697,  0.7836,  0.3497, -1.6995, -1.1404, -0.6248,
          -0.4275,  0.0626,  1.4395,  0.9323, -0.7109,  0.3818,  1.0305,
           0.8644,  1.1271],
         [-1.1540,  0.8253,  0.3173, -0.2124, -1.2695, -0.8238,  1.6244,
          -0.5813,  1.6354, -0.3787,  0.3473,  1.4297, -0.4350,  0.8373,
          -0.6544, -1.5077],
         [ 0.5574, -0.2438, -0.2493, -0.4500,  0.9800,  0.0817,  2.3445,
          -1.5691, -1.0391,  0.5847, -1.3261, -0.8119, -0.8042,  0.1226,
           1.2832,  0.5395],
         [-0.3151,  0.4882,  0.4471, -0.5274,  1.0980, -1.0861,  0.6844,
          -0.0609,  1.4091,  0.9468,  0.5419,  1.1426, -1.2990, -2.3474,
          -0.6565, -0.4658],
         [-1.3229,  0.1454, -0.1649, -0.5930,  0.1193, -0.6544,  1.4796,
          -2.0022,  0.0293,  0.1737,  1.2584,  1.0549, -1.0131, -0.2238,
           1.8410, -0.1273]],

        [[-1.6410, -0.1561, -0.0229,  0.0388, -1.9759,  0.7733,  1.3981,
           0.444